# 01. Azure Machine Learning 環境構築

**対応するテキスト**: [docs/03_AzureML環境構築.md](../docs/03_AzureML環境構築.md)

このノートブックで行うこと:

1. ワークスペースへの接続
2. コンピューティング クラスターの作成（`min_instances=0`）
3. カスタム環境（panda-gym + Stable-Baselines3）の作成
4. **疎通確認ジョブの実行と MLflow 記録の確認**

> ⚠ **実行前に必ず [docs/02_Azure環境の準備.md](../docs/02_Azure環境の準備.md) のチェックリストを完了してください。**
> 特に **権限**と **vCPU クォータ**が不足していると、この先すべて失敗します。

## 1. 必要なパッケージ

**Azure ML コンピューティング インスタンス上で実行する場合は、通常この手順は不要です**（導入済みのため）。
手元の PC で実行する場合のみ、次のセルのコメントを外して実行してください。

In [ ]:
# %pip install azure-ai-ml azure-identity mlflow azureml-mlflow

## 1-2. （Windows のみ）ローカルで RL 環境を動かす場合の準備

> [!NOTE]
> **この節は「手元の Windows PC で panda-gym を動かしたい人」だけが対象です。**
>
> - **Azure ML にジョブを投げて MLflow で結果を見るだけなら、この節は不要です。** 上の `azure-ai-ml` などだけで足ります。
> - Azure ML のコンピューティングは **Linux** なので、[../src/conda.yaml](../src/conda.yaml) 側にはこの手順は一切関係しません。
> - **ロボットが動く様子を 3D GUI で見る**（[04 章](../docs/04_RL環境を触って理解する.md) 4.6）のは、**ローカル実行でしかできません。** そのために必要なのがこの節です。

### なぜ `pip install panda-gym` だけでは Windows に入らないのか

`panda-gym` は物理エンジン **`pybullet`** に依存します。
この `pybullet` は、**PyPI に Windows 向けのビルド済みホイール（`win_amd64`）を公開していません。**
最新版 3.2.7 の配布物は `manylinux`（Linux）向けホイールとソース配布 (`.tar.gz`) だけです。

そのため Windows で `pip install panda-gym` を実行すると **ソースからのビルド**に入り、C++ コンパイラが無い環境では次のエラーで失敗します。

```
error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools":
https://visualstudio.microsoft.com/visual-cpp-build-tools/
```

一方 **conda-forge は `win-64` 向けのビルド済み `pybullet` を配布しています。**
そこで **`pybullet` だけを conda-forge から入れ、残りを `pip` で入れる**と、**C++ コンパイラなしで Windows に導入できます。**

> **出典（参考情報・サードパーティ）**
> - PyPI の `pybullet` 配布ファイル一覧（Windows 向けホイールが無いこと）: https://pypi.org/project/pybullet/#files
> - conda-forge の `pybullet` 配布パッケージ一覧（`win-64` を含むこと）: https://anaconda.org/conda-forge/pybullet
> - `panda-gym` の依存定義（`gymnasium>=0.26`, `pybullet`, `numpy<2`, `scipy`）: https://github.com/qgallouedec/panda-gym/blob/master/setup.py
> - Microsoft C++ Build Tools: https://visualstudio.microsoft.com/visual-cpp-build-tools/

### 手順

> [!IMPORTANT]
> **以下の①〜⑤は、ノートブックのセルではなくターミナル（PowerShell）で実行してください。**
> conda 環境の作成は、いま動いている Python カーネルの外側で行う作業だからです。

**① conda を用意する（未導入の場合）**

conda-forge を既定チャネルにした最小構成の **Miniforge** を使います。

- Windows 用インストーラー: `Miniforge3-Windows-x86_64.exe`
  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Windows-x86_64.exe
- インストール後は、スタート メニューの **「Miniforge Prompt」** から `conda` を使うのが最も確実です。

> [!WARNING]
> **インストール先のフォルダー名に空白や特殊文字を含めないでください。** Miniforge の公式 README が明示的に推奨しています。
> また **パスはできるだけ短くしてください**（理由は後述の「つまずきポイント」を参照）。
>
> 出典（参考情報・サードパーティ）: https://github.com/conda-forge/miniforge#windows

**② RL 用の環境を作る**

`pybullet` を **conda-forge のビルド済みパッケージ**から入れるのがポイントです。

```powershell
# pybullet だけ conda-forge から入れる（ここでコンパイラが不要になる）
conda create -y -n rl-local -c conda-forge python=3.10 pybullet "numpy<2" scipy pip

conda activate rl-local
```

**③ 残りを pip で入れる**

バージョンは [../src/conda.yaml](../src/conda.yaml)（Azure ML 側の定義）と揃えます。
`pybullet` は ② で導入済みなので、`pip` はここでビルドを試みません。

```powershell
pip install "gymnasium==0.29.1" "panda-gym==3.0.7" "stable-baselines3==2.4.1" imageio imageio-ffmpeg
```

**④ Azure ML を併用する場合は SDK も入れる**

```powershell
pip install azure-ai-ml azure-identity mlflow azureml-mlflow
```

**⑤ この環境を Jupyter カーネルとして登録する**

```powershell
pip install ipykernel
python -m ipykernel install --user --name rl-local --display-name "Python (rl-local)"
```

登録後、VS Code / Jupyter の右上でカーネル **「Python (rl-local)」** を選ぶと、
下の確認セルやノートブック [02_explore_rl_env.ipynb](02_explore_rl_env.ipynb) をこの環境で実行できます。

### 動作確認

次のセルを **`rl-local` カーネル**で実行してください。


In [ ]:
# ============================================================
#  （Windows のみ）ローカル RL 環境の動作確認
#  ※ このセルは "rl-local" カーネルで実行してください。
#  ※ Azure ML 上で実行する場合は不要です。
# ============================================================
import platform
import sys

import gymnasium as gym
import numpy as np
import panda_gym  # noqa: F401  # import すると Panda 系の環境が gymnasium に登録される

print("platform :", platform.system(), platform.machine())
print("python   :", sys.version.split()[0])
print("gymnasium:", gym.__version__)
print("numpy    :", np.__version__, "（panda-gym の制約により 2.x 未満である必要があります）")

# renderer="Tiny" は PyBullet の DIRECT 接続。ウィンドウを開かずに画像だけ取得する
env = gym.make("PandaReach-v3", render_mode="rgb_array", renderer="Tiny")
try:
    obs, info = env.reset(seed=0)
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    frame = np.asarray(env.render())

    print("observation shape:", obs["observation"].shape)
    print("action_space     :", env.action_space)
    print("render frame     :", frame.shape, frame.dtype)
finally:
    env.close()

print()
print("OK: ローカルで panda-gym が動作しています。")


### GUI（3D シミュレーター）が開くかを確認する

`render_mode="human"` にすると、PyBullet の **OpenGL ウィンドウが別ウィンドウとして開きます。**
PyBullet 公式ドキュメントによれば、**Linux と Windows では GUI が別スレッドで動作します**（macOS のみ OS の制約で同一スレッド）。

> **出典（参考情報・OSS 公式ソース）**: PyBullet Quickstart Guide
> https://github.com/bulletphysics/bullet3/blob/master/docs/pybullet_quickstart_guide/PyBulletQuickstartGuide.md.html
> 「The GUI connection will create a new graphical user interface (GUI) with 3D OpenGL rendering ... **On Linux and Windows this GUI runs in a separate thread**, while on OSX it runs in the same thread due to operating system limitations.」

> [!WARNING]
> - **GUI ウィンドウはノートブックの中ではなく、別ウィンドウとして開きます。** 画面の裏に隠れていないか確認してください。
> - **`env.close()` を必ず実行してください。** 実行しないとウィンドウとバックグラウンドのスレッドが残ります。
> - **リモート デスクトップや仮想マシンでは OpenGL 3 が使えず、GUI の起動に失敗することがあります。** その場合の詳細は [04 章](../docs/04_RL環境を触って理解する.md) 4.6 を参照してください。


In [ ]:
# ============================================================
#  （Windows のみ）GUI の動作確認
#  ※ 別ウィンドウが開きます。数秒でロボットが動いて自動的に閉じます。
# ============================================================
import time

import gymnasium as gym
import panda_gym  # noqa: F401

env = gym.make("PandaPickAndPlace-v3", render_mode="human")
obs, info = env.reset(seed=0)

try:
    # ランダムな行動で少しだけ動かす（まだ学習していないので動きはでたらめです）
    for _ in range(60):
        obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
        if terminated or truncated:
            obs, info = env.reset()

    # カメラ位置を変えられることの確認（詳細は docs/04 の 4.6.3）
    env.unwrapped.sim.place_visualizer(
        target_position=[0.0, 0.0, 0.0], distance=1.4, yaw=45, pitch=-30
    )
    time.sleep(2.0)
finally:
    env.close()  # ウィンドウとスレッドを必ず閉じる

print("OK: GUI が起動し、正常に終了しました。")


### つまずきポイント（Windows ローカル環境）

| 症状 | 原因 | 対処 |
|---|---|---|
| `error: Microsoft Visual C++ 14.0 or greater is required.` | `pybullet` を **PyPI から** 入れようとしてソースビルドに入った（Windows 向けホイールが無いため） | 上の手順②のとおり **`pybullet` を conda-forge から**入れる。または [Microsoft C++ Build Tools](https://visualstudio.microsoft.com/visual-cpp-build-tools/) を導入する |
| conda の環境作成が `Cannot find a valid extracted directory cache` / `remove_all: The directory is not empty.` で失敗する | **パスが長すぎる。** Windows は既定でファイル パスの上限が 260 文字。`pybullet` パッケージは深い階層のファイルを含むため上限に達しやすい | **conda 環境とパッケージ キャッシュを短いパスに置く**（例: `C:\conda\envs`）。または「長いパスを有効にする」ポリシーを有効化する |
| 手順③が非常に遅い | `stable-baselines3` が **PyTorch** を引き込むため（数百 MB のダウンロード） | 異常ではありません。完了まで待ってください |
| `numpy` が 2.x になっていて panda-gym が動かない | 後から入れたパッケージが numpy を 2.x に上げた | `pip install "numpy<2"` で戻す。`panda-gym` の `setup.py` が `numpy<2` を要求しています |
| GUI ウィンドウが開かない・真っ黒 | ウィンドウが背面に隠れている／リモート デスクトップや仮想マシンで OpenGL 3 が使えない | 画面を確認する。RDP・VM の場合は [04 章](../docs/04_RL環境を触って理解する.md) 4.6 を参照 |
| GUI を閉じてもプロセスが残る | `env.close()` を呼んでいない | 必ず `try` / `finally` で `env.close()` を呼ぶ |

> **出典（Microsoft Learn）**: [Maximum Path Length Limitation](https://learn.microsoft.com/windows/win32/fileio/maximum-file-path-limitation)
> 「the maximum length for a path is MAX_PATH, which is defined as 260 characters」／同ページに「Enable Long Paths in Windows 10, Version 1607, and Later」の有効化手順が記載されています。

> [!IMPORTANT]
> **ローカルと Azure ML では `pybullet` のバージョンが一致しません。**
>
> - **Azure ML 側**（[../src/conda.yaml](../src/conda.yaml)）は PyPI の `pybullet` を使います。
> - **Windows ローカル側**は conda-forge の `pybullet` を使います。両者はバージョン番号の付け方が異なります。
>
> ローカルは **「動きを目で見て理解する」「コードのバグを潰す」ため**の環境です。
> **実験結果として記録・比較するのは、必ず Azure ML 上で実行したジョブにしてください**（[05 章](../docs/05_ベースライン実験.md)）。

> [!NOTE]
> **この手順は次の組み合わせで実際に動作を確認しています。**
>
> | 項目 | 値 |
> |---|---|
> | OS | Windows (x64) |
> | Python | 3.10（conda-forge） |
> | `pybullet` | conda-forge の `win-64` ビルド済みパッケージ |
> | `panda-gym` | 3.0.7（pip） |
> | `gymnasium` | 0.29.1（pip） |
> | `stable-baselines3` | 2.4.1（pip） |
> | `torch` | 2.13.0+cpu（pip が自動解決） |
> | `numpy` | 1.26.4 |
>
> **確認できた動作**
>
> - 環境の生成 / `reset()` / `step()` / `render()`（`Tiny`）
> - **GUI（`human`）ウィンドウの起動と終了**、`place_visualizer()` によるカメラ移動
> - `make_vec_env` による並列環境の生成
> - **SAC + `HerReplayBuffer` による学習**、モデルの保存・読み込み、`predict(deterministic=True)` による推論


## 2. ワークスペース情報の入力

**取得方法**: [Azure ML studio](https://ml.azure.com) の右上にあるワークスペース名をクリックすると、
サブスクリプション ID・リソース グループ・ワークスペース名が表示されます。

> 出典: [Create an Azure Machine Learning compute cluster - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-create-attach-compute-cluster?view=azureml-api-2)

In [ ]:
# ============================================================
#  ここを自分の環境に書き換えてください
# ============================================================
SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

#  本ハンズオンで作成するリソースの名前
COMPUTE_NAME = "cpu-cluster"
COMPUTE_SIZE = "Standard_DS3_v2"   # CPU 4 コア。クォータに合わせて調整してください
MAX_INSTANCES = 4                    # 並列実行できるジョブ数の上限
ENVIRONMENT_NAME = "rl-panda-gym-env"

#  コスト集計用タグ（docs/02 の 2.6 を参照）
TAGS = {
    "project": "rl-workshop",
    "owner": "<your-alias>",
    "delete-after": "<YYYY-MM-DD>",
}

print("設定を読み込みました。")

## 3. ワークスペースへの接続

`DefaultAzureCredential` は複数の認証方法を順に試します。
コンピューティング インスタンス上では通常そのまま通ります。手元の PC では事前に `az login` が必要です。

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("Workspace      :", ws.name)
print("Location       :", ws.location)
print("Resource group :", ws.resource_group)

### ⚠ ここで失敗したら

| 症状 | 対処 |
|---|---|
| `DefaultAzureCredential failed to retrieve a token` | ターミナルで `az login`（ヘッドレスなら `az login --use-device-code`） |
| `ResourceNotFound` | サブスクリプション ID / リソース グループ名 / ワークスペース名の綴りを確認 |
| `AuthorizationFailed` | 権限不足。[docs/02](../docs/02_Azure環境の準備.md) の 2.1 に戻る |

## 4. コンピューティング クラスターの作成

**最重要のパラメーター**

| パラメーター | 意味 |
|---|---|
| `min_instances=0` | **ジョブが無い間はノード数 0 → コンピューティングの課金が止まる** |
| `max_instances` | 並列実行できるジョブ数の上限（クォータを超えないこと） |
| `idle_time_before_scale_down=120` | アイドル 120 秒でノードを解放する |

> 出典: [Create an Azure Machine Learning compute cluster - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-create-attach-compute-cluster?view=azureml-api-2)

In [ ]:
from azure.ai.ml.entities import AmlCompute

try:
    cluster = ml_client.compute.get(COMPUTE_NAME)
    print(f"既存のクラスターを使います: {cluster.name} (size={cluster.size}, max={cluster.max_instances})")
except Exception:
    print("クラスターが無いので作成します（数分かかります）...")
    cluster = AmlCompute(
        name=COMPUTE_NAME,
        type="amlcompute",
        size=COMPUTE_SIZE,
        min_instances=0,
        max_instances=MAX_INSTANCES,
        idle_time_before_scale_down=120,
        tags=TAGS,
    )
    cluster = ml_client.begin_create_or_update(cluster).result()
    print(f"作成しました: {cluster.name}")

## 5. カスタム環境の作成

[../src/conda.yaml](../src/conda.yaml) をもとに、**ベース Docker イメージ ＋ conda 環境**の形で作成します。

> ⚠ **最重要**: Azure ML は **conda 定義から新しい環境を作り、その中でジョブを実行します。**
> **ベースイメージに入っている Python パッケージは使えません。** 必要なものはすべて `conda.yaml` に書いてください。
>
> 出典: [Manage Azure Machine Learning environments with the CLI and SDK (v2) - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-manage-environments-v2?view=azureml-api-2)

> ⚠ **panda-gym は `numpy<2` を要求します。** `conda.yaml` で固定済みです。

In [ ]:
from azure.ai.ml.entities import Environment

#  ベースイメージ。Microsoft Learn の環境作成サンプルで使用されているものです。
#  もしこのイメージが取得できない場合は、docs/03 のトラブルシューティング #6 を参照してください。
BASE_IMAGE = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"

env = Environment(
    name=ENVIRONMENT_NAME,
    description="panda-gym + Stable-Baselines3 + MLflow (RL workshop)",
    image=BASE_IMAGE,
    conda_file="../src/conda.yaml",
    tags=TAGS,
)
env = ml_client.environments.create_or_update(env)

ENV_REF = f"{env.name}:{env.version}"
print("作成した環境:", ENV_REF)
print("※ 初回のイメージ構築には数分〜十数分かかります。studio の［環境］→［ビルド ログ］で進捗を確認できます。")

## 6. 疎通確認ジョブ

**ここまでの構築がすべて正しいかを 1 本のジョブで検証します。**

確認する項目:

1. panda-gym / Stable-Baselines3 が import できる
2. **`numpy<2` が守られている**
3. 環境を作って 1 エピソード動かせる
4. **MLflow にパラメーター・メトリック・成果物が記録される**

### 6-1. 疎通確認スクリプトを書き出す

In [ ]:
%%writefile ../src/smoke_test.py
"""Azure ML 疎通確認スクリプト。

Azure ML の Command Job として実行し、環境構築が正しいことを検証する。
ジョブとして実行される場合、MLflow の run は自動的に開始されるため
mlflow.start_run() は呼ばない。
  出典: https://learn.microsoft.com/azure/machine-learning/how-to-log-view-metrics?view=azureml-api-2
"""
import argparse
import json
import platform
import subprocess
import sys

import mlflow


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--env-id", type=str, default="PandaReach-v3")
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()

    import numpy as np
    import gymnasium as gym
    import panda_gym  # noqa: F401  import すると環境 ID が Gymnasium に登録される
    import stable_baselines3 as sb3

    # --- 1) バージョンを記録する（再現性のために最重要） ---
    mlflow.log_param("env_id", args.env_id)
    mlflow.log_param("seed", args.seed)
    mlflow.log_param("python_version", platform.python_version())
    mlflow.log_param("numpy_version", np.__version__)
    mlflow.log_param("gymnasium_version", gym.__version__)
    mlflow.log_param("panda_gym_version", panda_gym.__version__)
    mlflow.log_param("sb3_version", sb3.__version__)

    # panda-gym は setup.py で numpy<2 を要求している
    assert np.__version__.startswith("1."), f"panda-gym は numpy<2 を要求します (現在: {np.__version__})"

    # --- 2) 環境を作って中身を確認する ---
    #  "Tiny" は PyBullet のソフトウェア レンダラーで、GPU も X サーバーも不要。
    #  ヘッドレスな Azure ML のコンピューティングではこれが必須。
    #  panda-gym v3 の既定値も render_mode="rgb_array" / renderer="Tiny" だが、
    #  バージョン差で renderer 引数を受け付けない場合に備えてフォールバックする。
    try:
        env = gym.make(args.env_id, render_mode="rgb_array", renderer="Tiny")
    except TypeError as exc:
        print(f"[WARN] renderer 引数を渡せませんでした ({exc})。render_mode のみで再試行します。")
        env = gym.make(args.env_id, render_mode="rgb_array")

    #  gym.make はラッパーを返すが、.spec から登録時の max_episode_steps を参照できる。
    #  万が一取得できない場合は panda-gym の登録値 50 を使う。
    max_steps = getattr(env.spec, "max_episode_steps", None) or 50

    print("observation_space :", env.observation_space)
    print("action_space      :", env.action_space)
    print("max_episode_steps :", max_steps)
    mlflow.log_param("max_episode_steps", max_steps)
    mlflow.log_param("action_dim", int(env.action_space.shape[0]))

    # --- 3) ランダム方策で 1 エピソード動かす ---
    obs, info = env.reset(seed=args.seed)
    print("observation keys  :", sorted(obs.keys()))

    total_reward = 0.0
    steps = 0
    for step in range(max_steps):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        steps += 1
        mlflow.log_metric("step_reward", float(reward), step=step)
        if terminated or truncated:
            break

    # --- 4) 描画がヘッドレスで動くか確認する（評価動画の前提） ---
    frame = env.render()
    render_ok = frame is not None
    if render_ok:
        print("rendered frame shape:", np.asarray(frame).shape)
    mlflow.log_metric("render_ok", float(render_ok))
    env.close()

    mlflow.log_metric("episode_reward", total_reward)
    mlflow.log_metric("episode_length", steps)
    mlflow.log_metric("is_success", float(bool(info.get("is_success", False))))
    print(f"episode_reward={total_reward}  steps={steps}  is_success={info.get('is_success')}")

    # --- 5) 成果物を記録する ---
    freeze = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True
    ).stdout
    with open("pip_freeze.txt", "w", encoding="utf-8") as fp:
        fp.write(freeze)
    mlflow.log_artifact("pip_freeze.txt")

    summary = {
        "env_id": args.env_id,
        "episode_reward": total_reward,
        "episode_length": steps,
        "render_ok": render_ok,
    }
    with open("smoke_summary.json", "w", encoding="utf-8") as fp:
        json.dump(summary, fp, ensure_ascii=False, indent=2)
    mlflow.log_artifact("smoke_summary.json")

    print("SMOKE TEST OK")


if __name__ == "__main__":
    main()

### 6-2. ジョブを投入する

In [ ]:
from azure.ai.ml import command

smoke_job = command(
    code="../src",                       # このフォルダー全体がスナップショットとして保存される
    command="python smoke_test.py --env-id PandaReach-v3 --seed 0",
    environment=ENV_REF,
    compute=COMPUTE_NAME,
    experiment_name="rl-setup-check",
    display_name="smoke_test_pandareach",
    tags=TAGS,
)

returned_job = ml_client.jobs.create_or_update(smoke_job)
print("ジョブ名 :", returned_job.name)
print("studio  :", returned_job.studio_url)

### 6-3. ジョブの完了を待つ

初回は**イメージ構築のため十数分かかることがあります**。
上のセルで表示された studio の URL を開くと、進捗とログをリアルタイムで確認できます。

In [ ]:
ml_client.jobs.stream(returned_job.name)

job = ml_client.jobs.get(returned_job.name)
print("ステータス:", job.status)

## 7. MLflow に記録された内容を確認する

> ⚠ **初心者がハマる仕様**
> `run.data.metrics` は、同じ名前のメトリックについて **最後の値しか返しません。**
> 全ステップの値（学習曲線）が欲しい場合は **`MlflowClient.get_metric_history()`** を使ってください。
>
> 出典: [Log metrics, parameters, and files with MLflow - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-log-view-metrics?view=azureml-api-2)

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

#  Azure ML ワークスペースを MLflow のトラッキング先にする。
#  Azure ML のコンピューティング上で実行している場合は既に接続済みだが、
#  手元の PC から参照する場合は明示的な設定が必要。
#  出典: https://learn.microsoft.com/azure/machine-learning/how-to-use-mlflow-configure-tracking?view=azureml-api-2
try:
    tracking_uri = ml_client.workspaces.get(WORKSPACE_NAME).mlflow_tracking_uri
except AttributeError:
    #  フォールバック: 公式ドキュメントに記載されている形式で手動構築する
    #  ※ Private Link 有効のワークスペースでは URI 形式が異なるためこの方法は使えない
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
mlflow.set_tracking_uri(tracking_uri)
print("tracking uri:", tracking_uri[:60], "...")

#  Azure ML のジョブ名を渡すと、対応する MLflow の run が取得できる
run = mlflow.get_run(returned_job.name)
run_id = run.info.run_id   # 以降はこの run_id を使う

print("=== パラメーター ===")
for k, v in sorted(run.data.params.items()):
    print(f"  {k:22s}: {v}")

print("\n=== メトリック（最終値のみ） ===")
for k, v in sorted(run.data.metrics.items()):
    print(f"  {k:22s}: {v}")

client = MlflowClient()
print("\n=== 成果物 ===")
for a in client.list_artifacts(run_id):
    print(" ", a.path)

print("\n=== step_reward の履歴（先頭5件） ===")
history = client.get_metric_history(run_id, "step_reward")
for m in history[:5]:
    print(f"  step={m.step}  value={m.value}")
print(f"  ... 合計 {len(history)} 点")

## 8. ✅ チェックリスト（Azure 実験環境確認票）

**すべて満たしてから [docs/04_RL環境を触って理解する.md](../docs/04_RL環境を触って理解する.md) へ進んでください。**

- [ ] `ws.name` が正しく表示された
- [ ] コンピューティング クラスターが **`min_instances=0`** で作成できた
- [ ] カスタム環境のビルドが成功した
- [ ] 疎通確認ジョブのステータスが **`Completed`** になった
- [ ] `numpy_version` が **`1.x`** であることを確認した（`2.x` なら [docs/03](../docs/03_AzureML環境構築.md) の TS #8 へ）
- [ ] `render_ok` が **`1.0`** であることを確認した（ヘッドレス描画が動く＝評価動画を作れる）
- [ ] 成果物に `pip_freeze.txt` と `smoke_summary.json` が表示された
- [ ] `step_reward` の履歴が複数点取得できた

> **重要**: ここまで通れば、以降の演習は **「学習スクリプトを差し替えるだけ」** になります。

---

## ⚠ 後片付けのリマインド

**この日の作業を終えるときは、コンピューティング インスタンスを停止してください。**
クラスターは `min_instances=0` なので自動でノードが解放されますが、**インスタンスは手動停止（またはアイドル シャットダウン）が必要です。**

詳しい後片付け手順は [docs/09_評価・コスト・後片付け.md](../docs/09_評価・コスト・後片付け.md) にあります。